Data cleaning: data centre locations added using geocoding

In [49]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os
import json

load_dotenv('../../.env') 
api_key = os.getenv("OPENCAGE")

In [21]:
data_centres = pd.read_csv('../data/uk_data_centres.csv')

In [26]:
# get rid of generic address (eg. London, UK)
data_centres = data_centres[data_centres['Location'].apply(lambda x : x.count(',') > 1)]
# add coords columns
data_centres['Latitude'] = None
data_centres['Longitude'] = None
data_centres.drop(['Logo URL', 'Banner URL'], axis=1, inplace=True)

In [41]:
data_centres['Location']

0          Harbour Exchange Square, London, UK
2         1 Banbury Avenue, Slough SL1 4LH, UK
3            352 Buckingham Avenue, Slough, UK
4              8 Buckingham Avenue, Slough, UK
6                 9-17 Caxton Way, Watford, UK
                        ...                   
325     The Chubb Buildings, Wolverhampton, UK
326    Minton Place, Station Road, Swindon, UK
327               8 Buckingham Ave, Slough, UK
328      1 Harbour Exchange Square, London, UK
329       St Mark's Hill, London, Surbiton, UK
Name: Location, Length: 311, dtype: object

In [45]:
coordinates = []
for ind, location in enumerate(data_centres['Location']):
    url = 'https://api.opencagedata.com/geocode/v1/json?key=' + api_key + '&q=' + location
 
    response = requests.get(url)
    data = response.json()
    lat = data['results'][0]['geometry']['lat']
    lng = data['results'][0]['geometry']['lng']
    coordinates.append([lat, lng])

In [46]:
for ind, coord in enumerate(coordinates):
    data_centres['Latitude'].iloc[ind] = coord[0]
    data_centres['Longitude'].iloc[ind] = coord[1]


In [ ]:
features = []
# make json into geojson
for ind, row in data_centres.iterrows():
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row['Longitude'], row['Latitude']] 
        },
        "properties": {
            "Name": row["Name"],
            "Provider": row["Provider"],
            "Location": row["Location"],
            "Link": row["Link"]
        }
    }
    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open('../../data/geojson/data-centres-cleaned.geojson', 'w') as f:
    json.dump(geojson, f, indent=2)
